# Querying with SQL & Python

Connect to the Chinook SQLite sample database, inspect its schema, answer business questions with SQL, and chart a result.

**Database source:** [Chinook Database](https://github.com/lerocha/chinook-database). Run `python download_db.py` first; it saves the SQLite file to `data/Chinook_Sqlite.sqlite`.

In [ ]:
import sqlite3
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

DB_PATH = Path("data/Chinook_Sqlite.sqlite")
if not DB_PATH.exists():
    raise FileNotFoundError(
        f"Database not found at {DB_PATH}. Run `python download_db.py` from the project folder."
    )

connection = sqlite3.connect(DB_PATH)
print(f"Connected to {DB_PATH}")

## Explore the database schema

List every user table, then inspect its columns and declared foreign-key relationships. This shows how customers, invoices, tracks, and employees connect.

In [ ]:
tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type = 'table' AND name NOT LIKE 'sqlite_%' ORDER BY name",
    connection,
)
display(tables)

In [ ]:
schema_rows = []
relationship_rows = []

for table_name in tables["name"]:
    # Table names come from sqlite_master in the supplied sample database.
    columns = connection.execute(f'PRAGMA table_info("{table_name}")').fetchall()
    for column in columns:
        schema_rows.append(
            {
                "table": table_name,
                "column": column[1],
                "data_type": column[2],
                "not_null": bool(column[3]),
                "primary_key_position": column[5],
            }
        )

    foreign_keys = connection.execute(f'PRAGMA foreign_key_list("{table_name}")').fetchall()
    for foreign_key in foreign_keys:
        relationship_rows.append(
            {
                "table": table_name,
                "column": foreign_key[3],
                "references_table": foreign_key[2],
                "references_column": foreign_key[4],
            }
        )

schema_df = pd.DataFrame(schema_rows)
relationships_df = pd.DataFrame(
    relationship_rows,
    columns=["table", "column", "references_table", "references_column"],
)
display(schema_df)
display(relationships_df)

## 1. Which are the 10 best-selling tracks?

“Best-selling” is measured by total units purchased (`InvoiceLine.Quantity`). Revenue is shown too, and breaks ties in the sort order.

In [ ]:
top_tracks = pd.read_sql_query(
    '''
    SELECT
        t.TrackId AS track_id,
        t.Name AS track,
        ar.Name AS artist,
        SUM(il.Quantity) AS units_sold,
        ROUND(SUM(il.UnitPrice * il.Quantity), 2) AS revenue
    FROM InvoiceLine AS il
    JOIN Track AS t ON t.TrackId = il.TrackId
    LEFT JOIN Album AS al ON al.AlbumId = t.AlbumId
    LEFT JOIN Artist AS ar ON ar.ArtistId = al.ArtistId
    GROUP BY t.TrackId, t.Name, ar.Name
    ORDER BY units_sold DESC, revenue DESC, t.Name
    LIMIT 10
    ''',
    connection,
)
display(top_tracks)

## 2. Which country generates the most revenue?

Revenue is the sum of invoice totals grouped by billing country. The first row is the highest-revenue country.

In [ ]:
revenue_by_country = pd.read_sql_query(
    '''
    SELECT
        BillingCountry AS country,
        ROUND(SUM(Total), 2) AS revenue,
        COUNT(*) AS invoice_count
    FROM Invoice
    WHERE BillingCountry IS NOT NULL
    GROUP BY BillingCountry
    ORDER BY revenue DESC, country
    ''',
    connection,
)
display(revenue_by_country.head(10))
print("Top country:", revenue_by_country.iloc[0]["country"])

## 3. Who is the top-performing sales employee?

The Chinook schema assigns each customer to a support representative. Here, sales performance means the invoice revenue generated by each employee’s assigned customers.

In [ ]:
employee_sales = pd.read_sql_query(
    '''
    SELECT
        e.EmployeeId AS employee_id,
        e.FirstName || ' ' || e.LastName AS employee,
        ROUND(SUM(i.Total), 2) AS revenue,
        COUNT(i.InvoiceId) AS invoices,
        COUNT(DISTINCT c.CustomerId) AS customers
    FROM Employee AS e
    JOIN Customer AS c ON c.SupportRepId = e.EmployeeId
    JOIN Invoice AS i ON i.CustomerId = c.CustomerId
    GROUP BY e.EmployeeId, e.FirstName, e.LastName
    ORDER BY revenue DESC, employee
    ''',
    connection,
)
display(employee_sales)
print("Top sales employee:", employee_sales.iloc[0]["employee"])

## Visualize the top-selling tracks

A horizontal bar chart makes the longer track names easier to read.

In [ ]:
chart_data = top_tracks.sort_values("units_sold")
ax = chart_data.plot.barh(x="track", y="units_sold", figsize=(10, 6), legend=False, color="#e50914")
ax.set_title("Top 10 Best-Selling Chinook Tracks")
ax.set_xlabel("Units sold")
ax.set_ylabel("Track")
plt.tight_layout()
plt.show()

In [ ]:
connection.close()
print("Database connection closed.")